# SDF `to_plot_data()` — Comparison Notebook

This notebook demonstrates the new `to_plot_data()` method on `SDFData` / `SDFReactionData`
and compares the results with the existing `Coefficients.plot()` approach.

## 1. Setup — compute sensitivity data and create SDF

In [ ]:
import kika
from kika.sensitivities import compute_sensitivity, create_sdf_data
from kika.plotting import PlotBuilder

mctal_file = r'c:\Users\Usuario\BaradDur\Dev\kika\files\mcnp-mctal\PWRSphere.m'
input_file = r'c:\Users\Usuario\BaradDur\Dev\kika\files\mcnp-input\PWRSpherePERT.i'

# Compute sensitivity for Fe-56
sens = compute_sensitivity(inputfile=input_file, mctalfile=mctal_file, tally=4, zaid=26056, label='Fe-56, det1')

# Create SDF from the integral energy bin
sdf = create_sdf_data([sens], energy='integral', title='PWRSphere')
sdf

## 2. Old approach — `Coefficients.plot()`

The current way to plot sensitivity profiles: access via `SensitivityData` and call `.plot()` on each `Coefficients` object.

In [ ]:
# Plot elastic scattering (MT=2) using the old Coefficients.plot() approach
ax = sens.data['integral'][2].plot(xlog=True)
ax.set_title('Old approach: Coefficients.plot() — MT=2 (elastic)')

## 3. New approach (basic) — `SDFData.to_plot_data()` + `PlotBuilder`

Same data, plotted through the `to_plot_data()` → `PlotBuilder` workflow.

In [ ]:
# Get PlotData for elastic scattering (MT=2) by zaid+mt lookup
plot_data = sdf.to_plot_data(zaid=26056, mt=2)

_ = (
    PlotBuilder()
    .add_data(plot_data)
    .set_labels(
        title='New approach: to_plot_data() + PlotBuilder — MT=2 (elastic)',
        x_label='Energy (MeV)',
        y_label='Sensitivity per lethargy',
    )
    .set_scales(log_x=True)
    .build()
)

## 4. New approach (customised) — PlotBuilder styling

Demonstrate the customisation power of `PlotBuilder`: colours, log scale, labels, grid, font sizes, legend.

In [ ]:
plot_data = sdf.to_plot_data(zaid=26056, mt=2, color='crimson', linewidth=1.5)

_ = (
    PlotBuilder()
    .add_data(plot_data)
    .set_labels(
        title='Customised — Fe-56 elastic scattering',
        x_label='Energy (MeV)',
        y_label='Sensitivity per lethargy',
    )
    .set_scales(log_x=True)
    .set_grid(True)
    .set_font_sizes(title=14, labels=12)
    .build()
)

## 5. Multiple reactions overlay

Overlay several reactions on the same axes using `PlotBuilder`.

In [ ]:
builder = PlotBuilder()

# Add several reactions (without uncertainty to keep the plot clean)
for mt in [1, 2, 102]:
    try:
        pd = sdf.to_plot_data(zaid=26056, mt=mt, uncertainty=False)
        builder.add_data(pd)
    except ValueError:
        pass  # skip if reaction not in SDF

_ = (
    builder
    .set_labels(
        title='Fe-56 — multiple reactions',
        x_label='Energy (MeV)',
        y_label='Sensitivity per lethargy',
    )
    .set_scales(log_x=True)
    .build()
)

## 6. Side-by-side comparison

Old (`Coefficients.plot()`) vs New (`to_plot_data()` + `PlotBuilder`) on the same figure.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: old approach
sens.data['integral'][2].plot(ax=ax1, xlog=True)
ax1.set_title('Old: Coefficients.plot()')

# Right: new approach
plot_data = sdf.to_plot_data(zaid=26056, mt=2)
(
    PlotBuilder(ax=ax2)
    .add_data(plot_data)
    .set_labels(
        title='New: to_plot_data() + PlotBuilder',
        x_label='Energy (MeV)',
        y_label='Sensitivity per lethargy',
    )
    .set_scales(log_x=True)
    .build()
)

fig.tight_layout()

## 7. Access by index

Verify that `SDFData.to_plot_data(index=...)` works alongside the `(zaid, mt)` lookup.

In [ ]:
# Access the first reaction by index
pd_idx = sdf.to_plot_data(index=0, uncertainty=False)

_ = (
    PlotBuilder()
    .add_data(pd_idx)
    .set_labels(
        title=f'Access by index=0: {sdf.data[0].nuclide} {sdf.data[0].reaction_name}',
        x_label='Energy (MeV)',
        y_label='Sensitivity per lethargy',
    )
    .set_scales(log_x=True)
    .build()
)

## 8. Raw sensitivity (no per-lethargy normalisation)

Verify the `per_lethargy=False` option.

In [ ]:
pd_raw = sdf.to_plot_data(zaid=26056, mt=2, per_lethargy=False)

_ = (
    PlotBuilder()
    .add_data(pd_raw)
    .set_labels(
        title='Raw sensitivity (per_lethargy=False)',
        x_label='Energy (MeV)',
        y_label='Sensitivity coefficient',
    )
    .set_scales(log_x=True)
    .build()
)